## 1. Setup and Imports

In [ ]:
import os
import time
import math
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as F
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import DataLoader, Subset
from torch.optim import Adam
from tqdm import tqdm

# Для подсчёта FLOPs (пример: thop или fvcore)
# pip install thop (если не установлен)
try:
    from thop import profile, clever_format
except ImportError:
    print("thop not installed. Install with: pip install thop")

# Проверка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Фиксация seed для воспроизводимости
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

## 2. Implement LogMelFilterBanks Layer

In [ ]:
class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        # general params and params defined by the exercise
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.window = torch.hann_window(self.window_length)

        # TODO: инициализация STFT параметров
        # self.hop_length = ...
        # self.center = ...
        # self.return_complex = ...
        # self.onesided = ...
        # self.normalize_stft = ...
        # self.pad_mode = ...
        # self.power = ...

        # TODO: инициализация параметров mel filter banks
        # self.f_min_hz = ...
        # self.f_max_hz = ...
        # self.norm_mel = ...
        # self.mel_scale = ...

        # создание mel-банков (метод ниже)
        self.mel_fbanks = self._init_melscale_fbanks()

    def _init_melscale_fbanks(self):
        # TODO: вызвать F.melscale_fbanks с правильными аргументами
        # return F.melscale_fbanks(...)
        pass

    def spectrogram(self, x):
        # TODO: реализовать torch.stft с параметрами класса
        # return torch.stft(...)
        pass

    def forward(self, x):
        # TODO:
        # 1) вычислить спектрограмму через self.spectrogram(x)
        # 2) взять степень (power) → энергия спектра
        # 3) применить mel-фильтры (self.mel_fbanks)
        # 4) взять логарифм (добавить eps)
        # 5) вернуть тензор формы (batch, n_mels, n_frames)
        pass

## 3. Correctness check (vs torchaudio)

In [ ]:
# Загрузка тестового аудио (пример из датасета или любой файл 16 кГц)
# Вместо реального wav можно сгенерировать синусоиду, но для отчёта нужен реальный файл.
# Здесь используем синтетический сигнал для быстрой проверки, позже заменим на реальный.
def generate_test_signal(sr=16000, duration=1.0):
    t = torch.linspace(0, duration, int(sr*duration))
    freq = 440.0  # A4
    signal = 0.5 * torch.sin(2 * np.pi * freq * t)
    return signal.unsqueeze(0)  # (1, time)

signal, sr = generate_test_signal(), 16000
print("Test signal shape:", signal.shape)

# Инициализация нашей реализации и torchaudio MelSpectrogram
n_mels_test = 80
our_layer = LogMelFilterBanks(samplerate=sr, n_mels=n_mels_test)
mel_torch = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr,
    n_fft=400,
    hop_length=160,
    n_mels=n_mels_test,
    power=2.0
)

# Вычисление
with torch.no_grad():
    logmel_ours = our_layer(signal)  # (1, n_mels, frames)
    spec_torch = mel_torch(signal)   # (1, n_mels, frames) — уже мощность (power=2)
    logmel_torch = torch.log(spec_torch + 1e-6)

# Визуализация
plt.figure(figsize=(12, 5))
plt.subplot(1,2,1)
plt.imshow(logmel_ours[0].cpu().numpy(), aspect='auto', origin='lower')
plt.title('Our LogMelFilterBanks')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(logmel_torch[0].cpu().numpy(), aspect='auto', origin='lower')
plt.title('torchaudio MelSpectrogram + log')
plt.colorbar()
plt.tight_layout()
plt.savefig('logmel_comparison.png')
plt.show()

# Проверка на близость (assert)
assert torch.allclose(logmel_ours, logmel_torch, atol=1e-4), "Реализация не совпадает с эталоном"
print("Проверка пройдена: выходы совпадают.")

## 4. Подготовка датасета Speech Commands (бинарная классификация "yes"/"no")

In [ ]:
# Определим корневую папку для датасета (скачается при первом запуске)
data_dir = "./data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Загрузка датасета (torchaudio.datasets.SPEECHCOMMANDS)
# ВАЖНО: датасет большой, скачивание может занять время
# Используем стандартные сплиты:
#   "validation_list.txt" и "testing_list.txt" для валидации и теста,
#   остальное — тренировка.

# Функция для получения сплитов
def get_speechcommands_subsets(root, subset_ratio=1.0):
    # Загружаем весь датасет без трансформаций (возвращает (waveform, sample_rate, label, speaker_id, utterance_number))
    full_dataset = SPEECHCOMMANDS(root=root, download=True)

    # Читаем списки файлов для валидации и теста
    with open(os.path.join(root, "validation_list.txt"), "r") as f:
        val_files = set(f.read().splitlines())
    with open(os.path.join(root, "testing_list.txt"), "r") as f:
        test_files = set(f.read().splitlines())

    # Фильтруем только yes и no
    def filter_yes_no(item):
        label = item[2].lower()
        return label in ['yes', 'no']

    # Индексы, соответствующие датасету
    indices = []
    labels = []
    for i, (wave, sr, label, _, _) in enumerate(full_dataset):
        if label.lower() in ['yes', 'no']:
            indices.append(i)
            labels.append(0 if label.lower() == 'no' else 1)  # no=0, yes=1

    # Разбиваем по сплитам
    train_idx, val_idx, test_idx = [], [], []
    for i, idx in enumerate(indices):
        # Получаем путь относительно корня (для сверки со списками)
        # Формат строки в validation_list.txt: "yes/0ab3b47d_nohash_0.wav"
        file_path = full_dataset._walker[idx]  # внутри _walker хранится относительный путь
        if file_path in val_files:
            val_idx.append(idx)
        elif file_path in test_files:
            test_idx.append(idx)
        else:
            train_idx.append(idx)

    # Применяем subset_ratio (для быстрых экспериментов)
    if subset_ratio < 1.0:
        train_idx = train_idx[:int(len(train_idx) * subset_ratio)]
        val_idx = val_idx[:int(len(val_idx) * subset_ratio)]
        test_idx = test_idx[:int(len(test_idx) * subset_ratio)]

    # Создаем Subset'ы
    train_set = Subset(full_dataset, train_idx)
    val_set = Subset(full_dataset, val_idx)
    test_set = Subset(full_dataset, test_idx)

    # Вспомогательная функция для получения меток из Subset
    def get_labels(subset):
        lbls = []
        for i in range(len(subset)):
            _, _, lbl, _, _ = subset[i]
            lbls.append(0 if lbl.lower() == 'no' else 1)
        return torch.tensor(lbls)

    return train_set, val_set, test_set, get_labels(train_set), get_labels(val_set), get_labels(test_set)

# Используем 20% данных для ускорения отладки (можно убрать потом)
train_set, val_set, test_set, train_labels, val_labels, test_labels = get_speechcommands_subsets(data_dir, subset_ratio=0.2)
print(f"Train samples: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Class distribution: train no/yes = {train_labels.tolist().count(0)}/{train_labels.tolist().count(1)}")

# %%
# Функция коллата для даталоадера: применяет LogMelFilterBanks на лету
class AudioTransform:
    def __init__(self, mel_layer):
        self.mel_layer = mel_layer
    def __call__(self, waveform, sample_rate, label, speaker_id, utterance_number):
        # Убедимся, что семплирование 16 кГц (в датасете уже 16 кГц)
        # Применяем mel банки
        mel_spec = self.mel_layer(waveform.unsqueeze(0)).squeeze(0)  # (n_mels, T)
        return mel_spec, label

def collate_fn(batch):
    # batch: list of (mel_spec, label)
    # Паддинг по временной оси
    mels = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    # Находим максимальную длину
    max_len = max(m.shape[1] for m in mels)
    padded_mels = []
    for m in mels:
        pad_len = max_len - m.shape[1]
        if pad_len > 0:
            m = torch.nn.functional.pad(m, (0, pad_len))
        padded_mels.append(m)
    mels_stacked = torch.stack(padded_mels)  # (B, n_mels, T)
    labels = torch.tensor(labels, dtype=torch.long)
    return mels_stacked, labels

# Создадим даталоадеры (batch_size маленький для отладки)
BATCH_SIZE = 32
mel_layer = LogMelFilterBanks(samplerate=16000, n_mels=80)  # сначала с 80, позже будем менять
train_transform = AudioTransform(mel_layer)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)

# Проверка одного батча
batch_mels, batch_labels = next(iter(train_loader))
print("Batch mels shape:", batch_mels.shape)  # (B, n_mels, T)
print("Batch labels:", batch_labels)

## 5. Определение модели CNN (Conv1d) и вспомогательных функций


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_mels=80, n_classes=2, groups=1):
        super().__init__()
        # Пример архитектуры: несколько Conv1d с батчнормом и пулингом
        # Общее число параметров ~100K
        self.conv1 = nn.Conv1d(in_channels=n_mels, out_channels=32, kernel_size=3, stride=1, padding=1, groups=groups)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1, groups=groups)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, n_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # x: (B, n_mels, T)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1)  # (B, 64)
        x = self.dropout(x)
        x = self.fc(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_flops(model, input_shape=(1, 80, 101)):  # пример входного размера
    # Используем thop
    input_tensor = torch.randn(input_shape).to(next(model.parameters()).device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    return flops, params

# Пример
model = SimpleCNN(n_mels=80)
print(f"Model parameters: {count_parameters(model)}")
# flops, _ = compute_flops(model)
# print(f"FLOPs: {flops}")

## 6. Функции обучения и валидации

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

## 7. Эксперименты с количеством мел-фильтров (n_mels = 20, 40, 80)

In [ ]:
# TODO: реализовать цикл по n_mels, обучить модели, сохранить метрики и построить графики

n_mels_list = [20, 40, 80]
results_mels = {}

for n_mels in n_mels_list:
    print(f"\n=== Training with n_mels = {n_mels} ===")

    # Создаём новый слой Mel и даталоадеры (внимание: пересоздавать датасет каждый раз дорого, лучше динамически менять)
    # Для простоты создадим новый даталоадер с новым мел-слоем
    # (можно оптимизировать, но для отчёта пойдёт)
    mel_layer_exp = LogMelFilterBanks(samplerate=16000, n_mels=n_mels)
    train_transform_exp = AudioTransform(mel_layer_exp)
    train_loader_exp = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                                  collate_fn=collate_fn, num_workers=2)
    val_loader_exp = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                                collate_fn=collate_fn, num_workers=2)
    # Обновим модель с соответствующим входным каналом
    model = SimpleCNN(n_mels=n_mels, n_classes=2, groups=1).to(device)
    optimizer = Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    epochs = 10  # небольшое число для демонстрации, можно увеличить
    train_losses = []
    val_accs = []
    epoch_times = []

    for epoch in range(1, epochs+1):
        start_time = time.time()
        train_loss = train_epoch(model, train_loader_exp, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader_exp, criterion, device)
        epoch_time = time.time() - start_time
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time)
        print(f"Epoch {epoch:2d}: train_loss={train_loss:.4f}, val_acc={val_acc:.4f}, time={epoch_time:.2f}s")

    # Тестирование на тестовом наборе
    test_loss, test_acc = validate(model, test_loader_exp, criterion, device)
    print(f"Test accuracy: {test_acc:.4f}")

    results_mels[n_mels] = {
        'train_losses': train_losses,
        'val_accs': val_accs,
        'epoch_times': epoch_times,
        'test_acc': test_acc,
        'params': count_parameters(model)
    }

# Построение графиков (пример)
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
for n, res in results_mels.items():
    plt.plot(res['train_losses'], label=f'n_mels={n}')
plt.xlabel('Epoch')
plt.ylabel('Train loss')
plt.legend()
plt.subplot(1,3,2)
for n, res in results_mels.items():
    plt.plot(res['val_accs'], label=f'n_mels={n}')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend()
plt.subplot(1,3,3)
n_vals = list(results_mels.keys())
acc_vals = [results_mels[n]['test_acc'] for n in n_vals]
plt.bar([str(n) for n in n_vals], acc_vals)
plt.xlabel('n_mels')
plt.ylabel('Test accuracy')
plt.tight_layout()
plt.savefig('n_mels_experiment.png')
plt.show()

## 8. Эксперименты с параметром groups в Conv1d (baseline с n_mels=80)

In [ ]:
# TODO: зафиксировать лучший n_mels (или 80), менять groups = [1,2,4,8,16]

groups_list = [1, 2, 4, 8, 16]
results_groups = {}

best_n_mels = 80  # выберите лучший из предыдущего эксперимента
print(f"Using n_mels = {best_n_mels}")

# Пересоздаём даталоадеры для этого n_mels (ещё раз)
mel_layer_fixed = LogMelFilterBanks(samplerate=16000, n_mels=best_n_mels)
train_transform_fixed = AudioTransform(mel_layer_fixed)
train_loader_fixed = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                                collate_fn=collate_fn, num_workers=2)
val_loader_fixed = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=2)
test_loader_fixed = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=collate_fn, num_workers=2)

for groups in groups_list:
    print(f"\n=== Training with groups = {groups} ===")
    model = SimpleCNN(n_mels=best_n_mels, n_classes=2, groups=groups).to(device)
    optimizer = Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    epochs = 10
    train_losses = []
    val_accs = []
    epoch_times = []

    for epoch in range(1, epochs+1):
        start_time = time.time()
        train_loss = train_epoch(model, train_loader_fixed, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader_fixed, criterion, device)
        epoch_time = time.time() - start_time
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time)
        print(f"Epoch {epoch:2d}: train_loss={train_loss:.4f}, val_acc={val_acc:.4f}, time={epoch_time:.2f}s")

    test_loss, test_acc = validate(model, test_loader_fixed, criterion, device)
    print(f"Test accuracy: {test_acc:.4f}")
    params = count_parameters(model)
    # FLOPs (пример: размер входа зависит от временной размерности, возьмём среднюю)
    # В реальности нужно вычислить для типичного батча; здесь заглушка
    flops, _ = compute_flops(model, input_shape=(1, best_n_mels, 100))
    results_groups[groups] = {
        'train_losses': train_losses,
        'val_accs': val_accs,
        'epoch_times': epoch_times,
        'test_acc': test_acc,
        'params': params,
        'flops': flops
    }

# Построение графиков: epoch training time vs groups, params vs groups, flops vs groups
fig, axes = plt.subplots(1, 3, figsize=(15,4))
groups_sorted = sorted(groups_list)
# Среднее время эпохи (по последней эпохе или среднее по всем)
avg_time = [np.mean(results_groups[g]['epoch_times']) for g in groups_sorted]
axes[0].plot(groups_sorted, avg_time, marker='o')
axes[0].set_xlabel('groups')
axes[0].set_ylabel('Avg epoch time (s)')
axes[0].set_title('Training time vs groups')
params_vals = [results_groups[g]['params'] for g in groups_sorted]
axes[1].plot(groups_sorted, params_vals, marker='o')
axes[1].set_xlabel('groups')
axes[1].set_ylabel('Number of parameters')
axes[1].set_title('Params vs groups')
flops_vals = [results_groups[g]['flops'] for g in groups_sorted]
axes[2].plot(groups_sorted, flops_vals, marker='o')
axes[2].set_xlabel('groups')
axes[2].set_ylabel('FLOPs')
axes[2].set_title('FLOPs vs groups')
plt.tight_layout()
plt.savefig('groups_experiment.png')
plt.show()

## 9. Выводы

In [ ]:
#
# * Сравнение с эталонным torchaudio: ...
# * Влияние количества мел-фильтров на точность и время: ...
# * Влияние groups на скорость, число параметров и FLOPs: ...
# * Выбранная базовая модель и её характеристики.